# `aidp-snowflake` live test — Snowflake Spark connector
Requires `spark-snowflake_2.12-3.1.1.jar` + `snowflake-jdbc-3.19.0.jar` on the cluster classpath.

**Config:** non-sensitive options come from `assets/aidp/config.yaml`; `sfUser`/`sfPassword` come from the AIDP credential named under `snowflake.credential`. Edit `CONFIG_PATH` + `AIDP_ENV` at the top of the next cell.


In [6]:
# ══════════════════ Notebook config ══════════════════════════════════
# CONFIG_PATH : workspace path to assets/aidp/config.yaml. Get it from
#               the AIDP UI: right-click config.yaml → 'Copy path'.
# AIDP_ENV    : which block of config.yaml to load ('dev' or 'prod').
#               Defaults to 'dev'; Airflow can override via env var.
# ─────────────────────────────────────────────────────────────────────
import os

CONFIG_PATH = '/Workspace/sales-orders/assets/aidp/config.yaml'
AIDP_ENV    = os.environ.get('AIDP_ENV', 'dev')
print('env: ',AIDP_ENV)
# ═════════════════════════════════════════════════════════════════════

import sys
# Plugin scripts dir (uploaded once per workspace).
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


env:  dev


In [4]:
import yaml

with open(CONFIG_PATH) as f:
    _cfg_all = yaml.safe_load(f)

if AIDP_ENV not in _cfg_all:
    raise KeyError(f'env "{AIDP_ENV}" not in {CONFIG_PATH}; available: {sorted(_cfg_all)}')

cfg = _cfg_all[AIDP_ENV]
print(f'env    : {AIDP_ENV}')
print(f'config : {CONFIG_PATH}')

# Snowflake: merge non-sensitive options from config + user/password from AIDP secret.
# import aidputils
_snow_cfg     = cfg['snowflake']
_cred_name    = _snow_cfg['credential']
snow          = {k: v for k, v in _snow_cfg.items() if k != 'credential'}
snow['sfUser']     = aidputils.secrets.get(name=_cred_name, key='user')
snow['sfPassword'] = aidputils.secrets.get(name=_cred_name, key='password')
print(f'snow   : {snow["sfDatabase"]}.{snow["sfSchema"]} @ {snow["sfUrl"]} (cred={_cred_name})')
print('user: ',snow['sfUser'])
print('password: ',snow['sfPassword'])
source_table = cfg['source']['table']

df = (spark.read
      .format('snowflake')
      .options(**snow)
      .option('dbtable', source_table)
      .load())

display(df.limit(5))


env    : dev
config : /Workspace/sales-orders/assets/aidp/config.yaml


snow   : RAPPI_SANDBOX.SYNTH @ https://wfc22557.us-east-1.snowflakecomputing.com (cred=snowflake_dev)
user:  [REDACTED]
password:  [REDACTED]


In [5]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-snowflake',
    'env': AIDP_ENV,
    'auth': 'user-password (from AIDP secret)',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')


AIDP_LIVE_TEST_RESULT_BEGIN
{
  "connector": "aidp-snowflake",
  "env": "dev",
  "auth": "user-password (from AIDP secret)",
  "rows": 10100000,
  "schema": [
    "ACQUISITION_CHANNEL",
    "CANCELLATION_REASON",
    "CITY",
    "COUNTRY_CODE",
    "COURIER_ID",
    "CURRENCY",
    "DELIVERY_FEE",
    "DELIVERY_TIME_MINUTES",
    "DELIVERY_TYPE",
    "DISCOUNT",
    "FRAUD_RULE_TRIGGERED",
    "FRAUD_SCORE",
    "IS_FRAUDULENT",
    "IS_PRIME_USER",
    "IS_RECURRENT_ORDER",
    "LATITUDE",
    "LONGITUDE",
    "ORDER_CANCELLED_AT",
    "ORDER_CREATED_AT",
    "ORDER_DELIVERED_AT",
    "ORDER_ID",
    "ORDER_NUMBER_FOR_USER",
    "ORDER_STATUS",
    "PAYMENT_METHOD",
    "PRIME_TIER",
    "PRODUCT_AMOUNT",
    "SERVICE_FEE",
    "STORE_ID",
    "TIP",
    "TOTAL_AMOUNT",
    "USER_ID",
    "USER_SEGMENT",
    "VERTICAL"
  ],
  "timestamp_utc": 1779811585
}
AIDP_LIVE_TEST_RESULT_END
